# EDA Drill — Flotation Plant Recovery
**Scenario:** A mineral-processing plant runs 3 flotation lines. Each row is one hour of operation.
**Goal of the business:** understand what drives `recovery_pct` (% of metal recovered) before modeling.

**Rules:** fill every `____`. Each blank is one expression/name/argument. Hints are in comments.
Timebox: ~60 min. The dataset has **~10 planted problems** — find them all.

| Column | Meaning |
|---|---|
| timestamp | hourly reading |
| line_id | flotation line L1/L2/L3 |
| shift | Day/Night |
| ore_type | oxide / sulfide / mixed |
| ore_grade_pct | metal % in feed ore |
| feed_rate_tph | tonnes per hour |
| reagent_dose_gpt | collector dose, g/t |
| ph | slurry pH |
| temperature_c | slurry temp |
| particle_size_p80_um | grind size P80, microns |
| air_flow_m3min | flotation air |
| recovery_pct | **TARGET** |
| lab_assay_recovery_pct | lab-assayed recovery, reported next day |
| plant_id | plant code |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
TARGET = "recovery_pct"
df_raw = pd.read_csv("flotation_plant.csv")
df = df_raw.copy()          # never mutate the raw frame

## 1. First look — shape, dtypes, memory

In [ ]:
print(df.shape)                 # (rows, cols) — attribute, no ()
df.head(5)

In [ ]:
df.info()                       # dtypes + non-null counts in one call
# Q: which column that should be numeric is 'object'? Which date column is 'object'?

In [ ]:
df.describe(include="all").T   # transpose for readability
# Q: what looks impossible in min/max?

## 2. Data quality audit

### 2a. Duplicates

In [ ]:
n_dup = df.duplicated().sum()
print(f"exact duplicate rows: {n_dup}")
df = df.drop_duplicates().reset_index(drop=True)

### 2b. Fix dtypes

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

# feed_rate_tph is a string. Find values that are NOT parseable as numbers
bad = pd.to_numeric(df["feed_rate_tph"], errors="coerce").isna() & df["feed_rate_tph"].notna()
print(df.loc[bad, "feed_rate_tph"].value_counts())

In [ ]:
# Strip the unit text, then convert
df["feed_rate_tph"] = (df["feed_rate_tph"]
                       .str.replace(" tph", "", regex=False)
                       .pipe(pd.to_numeric))
df["feed_rate_tph"].dtype

### 2c. Constant / zero-information columns

In [ ]:
nunique = df.nunique()
const_cols = nunique[nunique <= 1].index.tolist()
print("constant columns:", const_cols)
df = df.drop(columns=const_cols)

### 2d. Categorical consistency

In [ ]:
cat_cols = df.select_dtypes(exclude=["number", "datetime"]).columns   # works on pandas 2 & 3
for c in cat_cols:
    print(c, df[c].unique())

In [ ]:
# normalize: strip whitespace + Title-case  ('day ' -> 'Day', 'NIGHT' -> 'Night')
for c in cat_cols:
    df[c] = df[c].str.strip().str.title()
df["ore_type"] = df["ore_type"].str.lower()   # keep ore_type lowercase
for c in cat_cols:
    print(c, df[c].value_counts().to_dict())

### 2e. Missing values — how much, and is it random?

In [ ]:
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss[miss > 0].round(2)

In [ ]:
# Is air_flow missingness random, or tied to a line / period?
df.groupby("line_id")["air_flow_m3min"].apply(lambda s: s.isna().sum())

In [ ]:
# When did L3's air_flow go missing?
m = df["air_flow_m3min"].isna() & (df["line_id"] == "L3")
df.loc[m, "timestamp"].agg(["min", "max", "count"])
# Q: MCAR, MAR or MNAR? How would you impute (or not)?

In [ ]:
# Rows with missing TARGET can't be used for supervised learning -> drop them
print("missing target:", df[TARGET].isna().sum())
df = df.dropna(subset=[TARGET])

### 2f. Impossible values & sentinels (domain checks)

In [ ]:
# Sentinel error codes are often -999 / 9999. Check the min of every numeric col
num_cols = df.select_dtypes(include="number").columns.drop(TARGET)
df[num_cols].min()

In [ ]:
# Replace sentinel with NaN, and enforce physical ranges
df.loc[df["feed_rate_tph"] < 0, "feed_rate_tph"] = np.nan
df.loc[~df["ph"].between(0, 14), "ph"] = np.nan
print((df["temperature_c"] > 60).sum(), "temps > 60C")   # plausible slurry temp < 60
df.loc[df["temperature_c"] > 60, "temperature_c"] = np.nan

### 2g. Stuck sensor (same value repeated many hours)

In [ ]:
# run-length of identical consecutive values
s = df["temperature_c"]
run_id = (s != s.shift()).cumsum()
run_len = s.groupby(run_id).transform("size")
stuck = run_len >= 10
print("stuck readings:", stuck.sum(), df.loc[stuck, "timestamp"].agg(["min", "max"]).tolist())
df.loc[stuck, "temperature_c"] = np.nan

## 3. Univariate analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.ravel(), list(num_cols) + [TARGET]):
    sns.histplot(df[c].dropna(), bins=40, ax=ax, kde=True)
    ax.set_title(f"{c}  skew={df[c].skew():.2f}")
plt.tight_layout()

In [ ]:
# IQR outlier count per numeric column
q1, q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
iqr = q3 - q1
out = ((df[num_cols] < q1 - 1.5 * iqr) | (df[num_cols] > q3 + 1.5 * iqr)).sum()
out.sort_values(ascending=False)
# Q: particle_size is right-skewed — what transform would you try?

## 4. Target vs. features

In [ ]:
# Correlation with target — Pearson AND Spearman (catches monotonic non-linear)
corr = pd.DataFrame({
    "pearson":  df[num_cols].corrwith(df[TARGET]),
    "spearman": df[num_cols].corrwith(df[TARGET], method="spearman"),
}).sort_values("pearson", key=abs, ascending=False)
corr.round(3)

In [ ]:
# 🚩 One feature correlates suspiciously well. Why?  (think: WHEN is it measured?)
LEAK = "lab_assay_recovery_pct"
num_cols = num_cols.drop(LEAK)

In [ ]:
# pH has ~0 Pearson corr but matters — check the shape
df["ph_bin"] = pd.qcut(df["ph"], q=10)
df.groupby("ph_bin", observed=True)[TARGET].mean().plot(marker="o", title="recovery vs pH decile")
# Q: what shape? what feature would capture it for a linear model?

In [ ]:
# Categorical drivers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["line_id", "ore_type", "shift"]):
    sns.boxplot(data=df, x=c, y=TARGET, ax=ax)
df.groupby("ore_type")[TARGET].agg(["mean", "std", "count"]).round(2)

In [ ]:
# Multicollinearity check among features
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)

## 5. Time structure (matters for the CV split later!)

In [ ]:
ts = df.set_index("timestamp").sort_index()
daily = ts[TARGET].resample("D").mean()
ax = daily.plot(alpha=0.5, label="daily")
daily.rolling(7).mean().plot(ax=ax, label="7-day MA")
ax.legend()

In [ ]:
# Recovery by line over time — is any line drifting?
ts.groupby("line_id")[TARGET].resample("W").mean().unstack(0).plot(title="weekly recovery by line")
# Q: given hourly time series data, would you use random KFold? What instead?

## 6. EDA summary — write it like you'd say it to the interviewer
Fill in 5–7 bullets:
1. Data quality issues found & fix applied:
2. Leakage:
3. Strongest drivers:
4. Non-linear effects:
5. Missingness pattern & imputation plan:
6. Validation strategy implication:
7. Next step (baseline model):

## ✅ Answer key — the 10 planted problems
1. **25 exact duplicate rows** → `drop_duplicates()`
2. **timestamp stored as string** → `pd.to_datetime`
3. **feed_rate_tph is text**, 30 rows say `"310 tph"` → strip unit, `to_numeric`
4. **Constant column** `plant_id` → drop (zero information)
5. **Messy categories**: `'day '`, `'NIGHT'`, `'Sulfide'` → strip + standardize case
6. **Missing values**: MCAR in reagent/pH/particle size; **block outage** of air_flow on L3, Feb 10–14 (MAR, tied to line + time) → impute within line, or add an `is_missing` flag; 20 missing targets → drop
7. **Sentinel** `-999` in feed_rate → NaN
8. **Impossible values**: pH 14.8; temperature ×10 decimal glitch (>60 °C) → NaN
9. **Stuck sensor**: temperature frozen at 24.3 for 61 h (Jan 26–28) → NaN
10. **Target leakage**: `lab_assay_recovery_pct` (r≈0.99) is reported the next day → exclude

**Signal findings:** ore grade is the top linear driver (+), then air flow (+), particle size (−), feed rate (−), reagent (+).
**pH ≈ 0 linear correlation but an inverted-U** (peak near 9.5) → add `(ph - 9.5)**2` or use trees.
Ore type: oxide < mixed < sulfide; L3 > L1 > L2.
**Validation:** hourly time series → `TimeSeriesSplit` / time-based holdout, not random KFold (neighboring hours are autocorrelated, so a random split leaks).
